# Analisis Tuntutan Perbelanjaan

Notebook ini menunjukkan cara untuk mencipta agen yang menggunakan pemalam untuk memproses perbelanjaan perjalanan daripada imej resit tempatan, menjana e-mel tuntutan perbelanjaan, dan memvisualisasikan data perbelanjaan menggunakan carta pai. Agen memilih fungsi secara dinamik berdasarkan konteks tugasan.

Langkah-langkah:
1. Agen OCR memproses imej resit tempatan dan mengekstrak data perbelanjaan perjalanan.
2. Agen E-mel menjana e-mel tuntutan perbelanjaan.

### Contoh senario perbelanjaan perjalanan:
Bayangkan anda seorang pekerja yang melakukan perjalanan untuk mesyuarat perniagaan di bandar lain. Syarikat anda mempunyai polisi untuk membayar balik semua perbelanjaan berkaitan perjalanan yang munasabah. Berikut adalah pecahan perbelanjaan perjalanan yang berpotensi:
- Pengangkutan:
Tiket penerbangan pergi balik dari bandar rumah anda ke bandar destinasi.
Perkhidmatan teksi atau penghantaran dan pengambilan dari lapangan terbang.
Pengangkutan tempatan di bandar destinasi (seperti pengangkutan awam, kereta sewa, atau teksi).

- Penginapan:
Penginapan hotel selama tiga malam di hotel perniagaan kelas pertengahan yang berhampiran dengan tempat mesyuarat.

- Makanan:
Elaun makanan harian untuk sarapan, makan tengah hari, dan makan malam, berdasarkan polisi per diem syarikat.

- Perbelanjaan Pelbagai:
Bayaran letak kereta di lapangan terbang.
Caj akses internet di hotel.
Tip atau caj perkhidmatan kecil.

- Dokumentasi:
Anda mengemukakan semua resit (penerbangan, teksi, hotel, makanan, dan lain-lain) dan laporan perbelanjaan yang lengkap untuk pembayaran balik.


## Import perpustakaan yang diperlukan

Import perpustakaan dan modul yang diperlukan untuk buku nota ini.


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import base64
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

 ## Define Expense Models

 Cipta model Pydantic untuk perbelanjaan individu dan kelas ExpenseFormatter untuk menukar pertanyaan pengguna ke dalam data perbelanjaan berstruktur.

 Setiap perbelanjaan akan diwakili dalam format:
 `{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## Mendefinisikan Alat - Menjana E-mel

Cipta fungsi alat untuk menjana e-mel bagi menghantar tuntutan perbelanjaan.  
- Alat ini menggunakan dekorator `@tool` daripada Microsoft Agent Framework.  
- Ia mengira jumlah keseluruhan perbelanjaan dan memformatkan maklumat tersebut ke dalam badan e-mel.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# Alat untuk Mengekstrak Perbelanjaan Perjalanan daripada Imej Resit

Cipta fungsi alat untuk mengekstrak perbelanjaan perjalanan daripada imej resit.  
- Alat ini menggunakan dekorator `@tool` daripada Microsoft Agent Framework.  
- Ia membaca imej resit, menukarnya kepada base64, dan mengembalikan URI data untuk ejen menganalisis.


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## Memproses Perbelanjaan

Tentukan ejen dan sambungkan mereka ke dalam aliran kerja berturutan menggunakan `WorkflowBuilder`.  
- Ejen OCR mengekstrak data perbelanjaan berstruktur dari imej resit menggunakan alat `load_receipt_image`.  
- Ejen Emel mengambil data yang diekstrak dan menghasilkan emel tuntutan perbelanjaan profesional menggunakan alat `generate_expense_email`.  
- `WorkflowBuilder` dengan `add_edge` mencipta saluran berturutan: Ejen OCR → Ejen Emel.


In [ ]:
ocr_agent = await provider.create_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = await provider.create_agent(
    name="EmailAgent",
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## Fungsi utama

Bina aliran kerja berurutan dan jalankan untuk memproses imej resit dan menjana e-mel tuntutan perbelanjaan.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Penafian**:
Dokumen ini telah diterjemahkan menggunakan perkhidmatan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Walaupun kami berusaha untuk ketepatan, sila ambil perhatian bahawa terjemahan automatik mungkin mengandungi kesilapan atau ketidaktepatan. Dokumen asal dalam bahasa asalnya harus dianggap sebagai sumber yang sahih. Untuk maklumat penting, terjemahan profesional oleh manusia adalah disyorkan. Kami tidak bertanggungjawab atas sebarang salah faham atau salah tafsir yang timbul daripada penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
